# YieldPredict — Kenali dataset

Jalankan sel dari atas ke bawah dengan kernel Python dari `.venv`. Di sini kita akan melihat contoh baris, ukuran dataset, dan tipe kolom.

## Baca CSV dan lihat lima baris pertama

`read_csv()` membaca file ke dalam DataFrame, yaitu tabel pandas. `head()` menampilkan lima baris pertama secara default.

In [2]:
from pathlib import Path

import pandas as pd

csv_path = Path("data/raw/yield_df.csv")
# Jika kernel berjalan dari notebooks/, cari data satu folder ke atas.
if not csv_path.is_file():
    csv_path = Path("..") / csv_path

df = pd.read_csv(csv_path)
df.head()

,Unnamed: 0,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,0,Albania,Maize,1990,36613,1485.0,121.0,16.37
1,1,Albania,Potatoes,1990,66667,1485.0,121.0,16.37
2,2,Albania,"Rice, paddy",1990,23333,1485.0,121.0,16.37
3,3,Albania,Sorghum,1990,12500,1485.0,121.0,16.37
4,4,Albania,Soybeans,1990,7000,1485.0,121.0,16.37


## Cek ukuran dataset

`shape` memberikan pasangan `(jumlah_baris, jumlah_kolom)`. `shape` adalah atribut, jadi ditulis tanpa tanda kurung.

In [3]:
df.shape

(28242, 8)

## Cek tipe kolom dan nilai terisi

Di keluaran `info()`, `Dtype` menunjukkan tipe data. `Non-Null Count` menunjukkan jumlah nilai yang terbaca sebagai terisi di setiap kolom. Bandingkan jumlah itu dengan total baris dari `shape`.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28242 entries, 0 to 28241
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Unnamed: 0                     28242 non-null  int64  
 1   Area                           28242 non-null  str    
 2   Item                           28242 non-null  str    
 3   Year                           28242 non-null  int64  
 4   hg/ha_yield                    28242 non-null  int64  
 5   average_rain_fall_mm_per_year  28242 non-null  float64
 6   pesticides_tonnes              28242 non-null  float64
 7   avg_temp                       28242 non-null  float64
dtypes: float64(3), int64(3), str(2)
memory usage: 2.1 MB


## Hasil cek awal

- CSV punya 28.242 baris dan 8 kolom. Salah satunya `Unnamed: 0`, yaitu kolom nomor urut dari file CSV.
- `Area` dan `Item` terbaca sebagai teks. `Year` dan target `hg/ha_yield` terbaca sebagai bilangan bulat. Curah hujan, pestisida, dan suhu terbaca sebagai bilangan desimal.
- Semua kolom berisi 28.242 nilai non-null menurut pandas. Kalau nanti ingin tahu apakah beberapa baris isinya berulang, itu perlu pemeriksaan tersendiri.

## Langkah 2 — Ringkas hasil panen per tanaman

Untuk ringkasan ini, satu catatan mewakili satu kombinasi negara (`Area`), tanaman (`Item`), dan tahun (`Year`). Kita ambil empat kolom terkait hasil panen dan hapus baris yang sama persis di tabel ringkasan. Dengan begitu, beberapa nilai suhu untuk panen yang sama tidak menambah bobotnya.

In [4]:
kunci_panen = ["Area", "Item", "Year"]
kolom_hasil = "hg/ha_yield"
catatan_panen = df[kunci_panen + [kolom_hasil]].drop_duplicates()

if catatan_panen.duplicated(subset=kunci_panen).any():
    raise ValueError("Ada kombinasi negara, tanaman, dan tahun dengan hasil panen berbeda.")

print(f"Jumlah baris CSV: {len(df)}")
print(f"Jumlah catatan hasil panen unik: {len(catatan_panen)}")

Jumlah baris CSV: 28242
Jumlah catatan hasil panen unik: 13130


### Baca median dan kuartil

Bagi hasil panen dengan 10.000 untuk mengubah `hg/ha` menjadi `ton/ha`. Lalu `groupby("Item")` membentuk satu kelompok untuk tiap tanaman dan `describe()` menghitung statistiknya.

Median adalah nilai tengah. Q1 dan Q3 adalah batas bawah dan atas untuk sekitar 50% catatan di tengah. Semua nilai hasil panen di tabel memakai `ton/ha`.

In [5]:
catatan_panen = catatan_panen.assign(yield_ton_ha=catatan_panen[kolom_hasil] / 10_000)
statistik_panen = catatan_panen.groupby("Item")["yield_ton_ha"].describe()

label_kolom = {
    "count": "Jumlah catatan",
    "min": "Minimum",
    "25%": "Q1",
    "50%": "Median",
    "75%": "Q3",
    "max": "Maksimum",
}
ringkasan_panen = statistik_panen[list(label_kolom)].rename(columns=label_kolom)
ringkasan_panen = ringkasan_panen.astype({"Jumlah catatan": int})
ringkasan_panen.round(3)

,Jumlah catatan,Minimum,Q1,Median,Q3,Maksimum
Item,,,,,,
Cassava,941,1.178,5.860,10.000,13.217,38.582
Maize,2028,0.085,1.407,2.449,5.319,20.756
Plantains and others,464,2.135,5.884,9.702,12.500,41.850
Potatoes,2091,0.841,11.502,16.189,23.602,50.141
"Rice, paddy",1502,0.203,2.287,3.355,4.893,10.390
Sorghum,1435,0.058,0.719,1.233,2.413,20.600
Soybeans,1242,0.005,1.000,1.620,2.221,4.161
Sweet potatoes,1163,0.880,5.409,8.750,14.997,40.000
Wheat,1810,0.171,1.589,2.432,4.129,9.939


### Contoh membaca baris padi

Untuk `Rice, paddy`, ada 1.502 catatan. Median sekitar 3,36 ton/ha, dan sekitar separuh catatan berada di antara 2,29 dan 4,89 ton/ha.

## Langkah 3 — Selidiki baris berulang

Mulai dari `Rice, paddy` di Indonesia tahun 2013. Saring baris dengan tiga kondisi sekaligus, lalu bandingkan target dan ketiga fitur numerik untuk melihat nilai mana yang berubah.

In [6]:
kolom_pembanding = [
    kolom_hasil,
    "average_rain_fall_mm_per_year",
    "pesticides_tonnes",
    "avg_temp",
]
padi_indonesia_2013 = df.loc[
    (df["Area"] == "Indonesia")
    & (df["Item"] == "Rice, paddy")
    & (df["Year"] == 2013),
    kolom_pembanding,
]
padi_indonesia_2013.sort_values("avg_temp").reset_index(drop=True)

,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,51520,2702.0,1597.0,25.48
1,51520,2702.0,1597.0,26.72
2,51520,2702.0,1597.0,27.30
3,51520,2702.0,1597.0,27.34
4,51520,2702.0,1597.0,27.37
5,51520,2702.0,1597.0,27.88


### Bedakan baris identik dan nilai yang berbeda

Abaikan `Unnamed: 0` saat membandingkan isi baris. Buat salinan dengan satu baris per isi yang unik, lalu pakai `groupby()` dan `nunique()` untuk menghitung berapa nilai berbeda yang muncul dalam setiap kombinasi negara, tanaman, dan tahun.

In [7]:
baris_unik = df[kunci_panen + kolom_pembanding].drop_duplicates()
nilai_per_panen = baris_unik.groupby(kunci_panen)[kolom_pembanding].nunique()
kolom_bervariasi = nilai_per_panen.gt(1)

print(f"Baris tambahan yang isinya identik: {len(df) - len(baris_unik)}")
print(f"Kombinasi dengan nilai berbeda: {kolom_bervariasi.any(axis=1).sum()}")
kolom_bervariasi.sum().to_frame("Jumlah kombinasi dengan lebih dari satu nilai")

Baris tambahan yang isinya identik: 2310
Kombinasi dengan nilai berbeda: 3690


,Jumlah kombinasi dengan lebih dari satu nilai
hg/ha_yield,0
average_rain_fall_mm_per_year,0
pesticides_tonnes,0
avg_temp,3690


### Bandingkan kumpulan suhu antar tanaman

Urutkan nilai suhu unik untuk tiap tanaman supaya kumpulannya bisa dibandingkan. Lalu cek apakah semua tanaman dalam negara dan tahun yang sama mendapat kumpulan suhu yang sama.

In [8]:
suhu_per_panen = (
    baris_unik.groupby(kunci_panen)["avg_temp"]
    .agg(lambda suhu: tuple(sorted(suhu.unique().tolist())))
    .reset_index(name="nilai_suhu_unik")
)
suhu_per_negara_tahun = suhu_per_panen.groupby(["Area", "Year"])["nilai_suhu_unik"].agg(
    jumlah_tanaman="size", jumlah_kumpulan_suhu="nunique"
)
lintas_tanaman = suhu_per_negara_tahun.loc[suhu_per_negara_tahun["jumlah_tanaman"] > 1]
jumlah_berbeda = lintas_tanaman["jumlah_kumpulan_suhu"].gt(1).sum()
print(f"Negara-tahun dengan lebih dari satu tanaman: {len(lintas_tanaman)}")
print(f"Negara-tahun dengan kumpulan suhu berbeda antar tanaman: {jumlah_berbeda}")

suhu_per_panen.loc[
    (suhu_per_panen["Area"] == "Indonesia") & (suhu_per_panen["Year"] == 2013),
    ["Item", "nilai_suhu_unik"],
].reset_index(drop=True)

Negara-tahun dengan lebih dari satu tanaman: 2233
Negara-tahun dengan kumpulan suhu berbeda antar tanaman: 0


,Item,nilai_suhu_unik
0,Cassava,"(25.48, 26.72, 27.3, 27.34, 27.37, 27.88)"
1,Maize,"(25.48, 26.72, 27.3, 27.34, 27.37, 27.88)"
2,Potatoes,"(25.48, 26.72, 27.3, 27.34, 27.37, 27.88)"
3,"Rice, paddy","(25.48, 26.72, 27.3, 27.34, 27.37, 27.88)"
4,Soybeans,"(25.48, 26.72, 27.3, 27.34, 27.37, 27.88)"
5,Sweet potatoes,"(25.48, 26.72, 27.3, 27.34, 27.37, 27.88)"


### Apa yang bisa disimpulkan?

Ada **2.310 baris tambahan yang identik**. Pada **3.690 kombinasi negara, tanaman, dan tahun**, `avg_temp` punya beberapa nilai, sedangkan hasil panen, curah hujan, dan pestisida tetap sama. Enam tanaman Indonesia tahun 2013 berbagi enam nilai suhu yang sama.

Pola ini konsisten dengan penggabungan data suhu berdasarkan negara dan tahun. Jika tabel suhu punya enam baris untuk satu negara dan tahun, penggabungan akan memasangkan setiap catatan panen dengan keenam baris itu. CSV gabungan hanya memberi negara dan tahun untuk menelusuri nilai suhu. Metadata pengukuran diperlukan untuk menilai apakah satu angka bisa mewakili seluruh negara dan tahun. [Dataset di Kaggle](https://www.kaggle.com/datasets/patelris/crop-yield-prediction-dataset?select=temp.csv)

## Langkah 4 — Siapkan tabel model awal

File `temp.csv` yang diunduh langsung dari Kaggle berisi 71.311 baris dengan tiga kolom, yaitu `year`, `country`, dan `avg_temp`. Ada 7.994 pasangan negara dan tahun dengan lebih dari satu baris. Enam nilai suhu untuk Indonesia tahun 2013 cocok dengan nilai yang muncul di CSV gabungan. [Sumber `temp.csv` di Kaggle](https://www.kaggle.com/datasets/patelris/crop-yield-prediction-dataset?select=temp.csv)

File suhu memberi negara dan tahun, sementara lokasi yang lebih rinci dan periode pengukuran dalam tahun masih belum diketahui. Untuk tabel awal, keluarkan `avg_temp` sehingga kita bisa memakai satu catatan per negara, tanaman, dan tahun tanpa mengasumsikan cara merata-ratakan suhu.

### Pilih lima fitur dan pisahkan target

Pilih negara, tanaman, tahun, curah hujan, penggunaan pestisida, dan hasil panen. Setelah itu, `drop_duplicates()` menyisakan satu baris untuk setiap isi yang identik pada kolom terpilih. Cek lagi kunci negara–tanaman–tahun supaya perbedaan nilai, jika ada, bisa diperiksa sebelum membuat model.

`fitur_model` adalah input yang dipakai model. `target_model` berisi `hg/ha_yield` dalam satuan asli `hg/ha`, yaitu nilai yang ingin diprediksi.

In [9]:
daftar_fitur = kunci_panen + ["average_rain_fall_mm_per_year", "pesticides_tonnes"]
data_model_awal = df[daftar_fitur + [kolom_hasil]].drop_duplicates(ignore_index=True)

if data_model_awal.duplicated(subset=kunci_panen).any():
    raise ValueError("Ada kombinasi negara, tanaman, dan tahun dengan nilai yang berbeda.")

fitur_model = data_model_awal[daftar_fitur]
target_model = data_model_awal[kolom_hasil]

print(f"Ukuran tabel model: {data_model_awal.shape}")
print(f"Ukuran fitur: {fitur_model.shape}")
print(f"Jumlah nilai target: {len(target_model)}")
data_model_awal.head()

Ukuran tabel model: (13130, 6)
Ukuran fitur: (13130, 5)
Jumlah nilai target: 13130


,Area,Item,Year,average_rain_fall_mm_per_year,pesticides_tonnes,hg/ha_yield
0,Albania,Maize,1990,1485.0,121.0,36613
1,Albania,Potatoes,1990,1485.0,121.0,66667
2,Albania,"Rice, paddy",1990,1485.0,121.0,23333
3,Albania,Sorghum,1990,1485.0,121.0,12500
4,Albania,Soybeans,1990,1485.0,121.0,7000


## Langkah 5 — Bagi data berdasarkan tahun

Pakai 1990–2006 untuk melatih model, 2007–2009 untuk membandingkan model, dan 2010–2013 untuk evaluasi akhir. Urutan ini meniru situasi ketika kita belajar dari data lama lalu memprediksi tahun yang lebih baru.

`between()` memilih baris di dalam rentang tahun, termasuk kedua batasnya. Setelah membagi data, cek apakah total catatan tetap sama dengan ukuran tabel model.

In [10]:
tahun = data_model_awal["Year"]
data_latih = data_model_awal.loc[tahun.between(1990, 2006)]
data_validasi = data_model_awal.loc[tahun.between(2007, 2009)]
data_uji = data_model_awal.loc[tahun.between(2010, 2013)]

pembagian_data = {
    "Train": data_latih,
    "Validation": data_validasi,
    "Test": data_uji,
}
ringkasan_split = pd.DataFrame([
    {
        "Bagian": nama,
        "Tahun awal": bagian["Year"].min(),
        "Tahun akhir": bagian["Year"].max(),
        "Jumlah catatan": len(bagian),
    }
    for nama, bagian in pembagian_data.items()
]).set_index("Bagian")

if ringkasan_split["Jumlah catatan"].sum() != len(data_model_awal):
    raise ValueError("Ada catatan di luar rentang tahun pembagian.")

ringkasan_split

,Tahun awal,Tahun akhir,Jumlah catatan
Bagian,,,
Train,1990,2006,9027
Validation,2007,2009,1748
Test,2010,2013,2355


## Langkah 6 — Ukur baseline median

Hitung median hasil panen dari **train**. Pakai angka tersebut untuk menebak setiap hasil panen di **validation**. Median adalah tebakan konstan yang meminimalkan rata-rata error absolut pada data yang dipakai untuk memilihnya.

MAE (*mean absolute error*) adalah rata-rata jarak antara prediksi dan hasil aktual, tanpa memperhitungkan arah selisihnya. `mean_absolute_error()` menghitungnya dalam `hg/ha` karena target dan prediksi memakai satuan itu. Bagi 10.000 untuk menampilkannya dalam `ton/ha`.

In [11]:
from sklearn.metrics import mean_absolute_error

target_latih = data_latih[kolom_hasil]
target_validasi = data_validasi[kolom_hasil]

median_latih = target_latih.median()
prediksi_baseline = pd.Series(median_latih, index=target_validasi.index)
mae_baseline_ton_ha = mean_absolute_error(target_validasi, prediksi_baseline) / 10_000

print(f"Prediksi median latih: {median_latih / 10_000:.3f} ton/ha")
print(f"MAE baseline di validation: {mae_baseline_ton_ha:.3f} ton/ha")

Prediksi median latih: 3.683 ton/ha
MAE baseline di validation: 5.732 ton/ha


MAE baseline menjadi patokan untuk Random Forest. Ukur keduanya pada 1.748 catatan validation yang sama. MAE yang lebih kecil berarti prediksi rata-rata lebih dekat ke hasil aktual.

## Langkah 7 — Latih Random Forest pertama

`OneHotEncoder` mengubah `Area` dan `Item` menjadi kolom indikator. Tiga fitur numerik diteruskan apa adanya. `handle_unknown="ignore"` membuat pipeline tetap bisa memproses kategori yang baru muncul di validation.

Pipeline menjalankan encoding dan pelatihan dalam satu `fit()`, memakai data train saja. Random Forest memakai 100 pohon dan merata-ratakan prediksi mereka. `random_state=42` menetapkan seed supaya percobaannya bisa diulang.

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

fitur_latih = data_latih[daftar_fitur]
preprocessing_awal = ColumnTransformer(
    [("kategori", OneHotEncoder(handle_unknown="ignore"), ["Area", "Item"])],
    remainder="passthrough",
)
pipeline_awal = Pipeline([
    ("preprocessing", preprocessing_awal),
    ("model", RandomForestRegressor(n_estimators=100, random_state=42)),
])

pipeline_awal.fit(fitur_latih, target_latih)
print(f"Jumlah catatan yang dipakai untuk training: {len(fitur_latih)}")

Jumlah catatan yang dipakai untuk training: 9027


### Bandingkan MAE pada validation

Pakai `predict()` untuk menerapkan pipeline yang sudah dilatih ke fitur validation. Bandingkan hasil prediksi dengan target validation, lalu hitung MAE dalam `ton/ha` seperti pada baseline.

In [13]:
fitur_validasi = data_validasi[daftar_fitur]
prediksi_random_forest = pipeline_awal.predict(fitur_validasi)
mae_random_forest_ton_ha = mean_absolute_error(target_validasi, prediksi_random_forest) / 10_000

ringkasan_model = pd.DataFrame({
    "Model": ["Baseline median", "Random Forest"],
    "MAE validation (ton/ha)": [mae_baseline_ton_ha, mae_random_forest_ton_ha],
}).set_index("Model")
ringkasan_model.round(3)

,MAE validation (ton/ha)
Model,
Baseline median,5.732
Random Forest,0.983


### Tambah pembanding dari riwayat panen

Hitung median hasil panen untuk tiap pasangan negara dan tanaman di train. Kalau pasangan di validation belum pernah muncul di train, pakai median jenis tanaman itu dari semua negara di train. Kalau jenis tanamannya juga baru, pakai median global dari Langkah 6.

`join()` memasangkan baris validation dengan median berdasarkan `Area` dan `Item`. `validate="many_to_one"` memastikan banyak baris validation boleh memakai median yang sama, sementara tiap pasangan punya satu nilai median dari train. Ukur MAE pada validation yang sama dengan Random Forest.

In [14]:
def prediksi_median_riwayat(data_referensi, fitur_prediksi, kolom_target):
    kunci_riwayat = ["Area", "Item"]
    median_per_pasangan = (
        data_referensi.groupby(kunci_riwayat)[kolom_target].median().rename("median_riwayat")
    )
    median_per_tanaman = data_referensi.groupby("Item")[kolom_target].median()
    median_global = data_referensi[kolom_target].median()

    riwayat = fitur_prediksi[kunci_riwayat].join(
        median_per_pasangan, on=kunci_riwayat, validate="many_to_one"
    )["median_riwayat"]
    median_tanaman = fitur_prediksi["Item"].map(median_per_tanaman)
    prediksi = riwayat.fillna(median_tanaman).fillna(median_global)
    return prediksi, int(riwayat.notna().sum())


prediksi_baseline_riwayat, jumlah_pasangan_validasi = prediksi_median_riwayat(
    data_latih, fitur_validasi, kolom_hasil
)
mae_baseline_riwayat_ton_ha = (
    mean_absolute_error(target_validasi, prediksi_baseline_riwayat) / 10_000
)

print(f"Catatan validation dengan riwayat pasangan: {jumlah_pasangan_validasi}/{len(data_validasi)}")
kolom_mae_validasi = "MAE validation (ton/ha)"
ringkasan_model_dengan_riwayat = ringkasan_model.copy()
ringkasan_model_dengan_riwayat.loc["Baseline median negara-tanaman", kolom_mae_validasi] = (
    mae_baseline_riwayat_ton_ha
)
ringkasan_model_dengan_riwayat.sort_values(kolom_mae_validasi).round(3)

Catatan validation dengan riwayat pasangan: 1748/1748


,MAE validation (ton/ha)
Model,
Random Forest,0.983
Baseline median negara-tanaman,1.563
Baseline median,5.732


### Cek MAE per jenis tanaman di validation

Hitung error absolut tiap prediksi dalam ton/ha. Lalu kelompokkan catatan berdasarkan `Item` dan ambil rata-rata error untuk setiap tanaman. Kedua model memakai catatan validation yang sama.

Selisih dihitung sebagai MAE baseline negara–tanaman dikurangi MAE Random Forest. Angka positif berarti Random Forest punya MAE lebih kecil. Tabel diurutkan dari MAE Random Forest yang terbesar, dengan jumlah catatan untuk memberi konteks saat membandingkan antar tanaman.

In [15]:
prediksi_rf_validasi = pd.Series(prediksi_random_forest, index=fitur_validasi.index)
error_validasi = data_validasi[["Item"]].assign(
    error_baseline_ton_ha=(target_validasi - prediksi_baseline_riwayat).abs() / 10_000,
    error_rf_ton_ha=(target_validasi - prediksi_rf_validasi).abs() / 10_000,
)

mae_per_tanaman = error_validasi.groupby("Item").agg(
    jumlah_catatan=("Item", "size"),
    mae_baseline_ton_ha=("error_baseline_ton_ha", "mean"),
    mae_rf_ton_ha=("error_rf_ton_ha", "mean"),
)
mae_per_tanaman["selisih_ton_ha"] = (
    mae_per_tanaman["mae_baseline_ton_ha"] - mae_per_tanaman["mae_rf_ton_ha"]
)

mae_per_tanaman.sort_values("mae_rf_ton_ha", ascending=False).round(3).rename(columns={
    "jumlah_catatan": "Jumlah catatan",
    "mae_baseline_ton_ha": "MAE baseline (ton/ha)",
    "mae_rf_ton_ha": "MAE Random Forest (ton/ha)",
    "selisih_ton_ha": "Selisih (ton/ha)",
})

,Jumlah catatan,MAE baseline (ton/ha),MAE Random Forest (ton/ha),Selisih (ton/ha)
Item,,,,
Potatoes,282,3.362,2.149,1.213
Cassava,123,2.700,1.666,1.035
Sweet potatoes,153,2.350,1.617,0.733
Plantains and others,60,3.783,1.506,2.277
Yams,60,1.878,1.051,0.827
Maize,270,1.132,0.729,0.402
"Rice, paddy",197,0.810,0.469,0.341
Wheat,242,0.565,0.416,0.149
Sorghum,190,0.484,0.362,0.122


Random Forest punya MAE lebih rendah pada kesepuluh tanaman di validation. `Plantains and others` mengalami penurunan terbesar, dari 3,783 ke 1,506 ton/ha pada 60 catatan. `Potatoes` masih punya MAE absolut tertinggi di antara tanaman yang diuji, yaitu 2,149 ton/ha pada 282 catatan.

## Langkah 8 — Evaluasi akhir di test 2010–2013

Pakai konfigurasi yang dipilih pada validation, yaitu Random Forest dengan 100 pohon, `random_state=42`, dan pipeline encoding yang sama. `clone()` menyalin konfigurasinya, lalu `fit()` melatih ulang encoder dan model dari awal pada gabungan train dan validation (1990–2009).

In [16]:
from sklearn.base import clone

data_latih_final = pd.concat([data_latih, data_validasi])
fitur_latih_final = data_latih_final[daftar_fitur]
target_latih_final = data_latih_final[kolom_hasil]

pipeline_final = clone(pipeline_awal)
pipeline_final.fit(fitur_latih_final, target_latih_final)
print(f"Jumlah catatan untuk pelatihan akhir: {len(data_latih_final)}")

Jumlah catatan untuk pelatihan akhir: 10775


### Ukur MAE pada 2.355 catatan test

Random Forest dan baseline median negara–tanaman memakai data referensi yang sama sampai 2009. Ukur keduanya pada test 2010–2013 dalam ton/ha. Periode test ini menentukan cakupan skor akhir yang dicantumkan di laporan UAS.

In [17]:
fitur_uji = data_uji[daftar_fitur]
target_uji = data_uji[kolom_hasil]

prediksi_rf_uji = pipeline_final.predict(fitur_uji)
prediksi_baseline_uji, jumlah_pasangan_uji = prediksi_median_riwayat(
    data_latih_final, fitur_uji, kolom_hasil
)
mae_rf_uji_ton_ha = mean_absolute_error(target_uji, prediksi_rf_uji) / 10_000
mae_baseline_uji_ton_ha = mean_absolute_error(target_uji, prediksi_baseline_uji) / 10_000

print(f"Catatan test dengan riwayat pasangan: {jumlah_pasangan_uji}/{len(data_uji)}")
ringkasan_uji = pd.DataFrame({
    "Model": ["Baseline median negara-tanaman", "Random Forest"],
    "MAE test (ton/ha)": [mae_baseline_uji_ton_ha, mae_rf_uji_ton_ha],
}).set_index("Model")
ringkasan_uji.round(3)

Catatan test dengan riwayat pasangan: 2329/2355


,MAE test (ton/ha)
Model,
Baseline median negara-tanaman,1.806
Random Forest,1.139


Random Forest mencapai **MAE 1,139 ton/ha** pada test 2010–2013, sekitar **36,9% lebih rendah** dari baseline riwayat panen yang mencapai **1,806 ton/ha**. Baseline memakai median negara–tanaman untuk 2.329 catatan dan median jenis tanaman untuk 26 catatan lainnya.